In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

CUDA available: True
GPU: Tesla T4


In [6]:
%cd /content/OttoLatin

!pwd
!ls
!ls data/eval


/content/OttoLatin
/content/OttoLatin
data  docs  README.md  scripts
fewshot.jsonl  naskh.jsonl  ot2tr.jsonl  riqa.jsonl  test_2k.jsonl


In [8]:
!pip install -q datasets pandas pyarrow matplotlib transformers accelerate peft sentencepiece


In [9]:
%cd /content/OttoLatin

!python scripts/step1_explore.py --out out/step1


/content/OttoLatin
[i] Downloading from Hugging Face: Kotek37/ottoman-turkish-4m

data/train-00000-of-00002.parquet: downloading bytes:  92% 190M/206M [00:03<00:00, 111MB/s, 15.9MB/s  ]
data/train-00000-of-00002.parquet: downloading bytes:  99% 205M/206M [00:03<00:00, 84.7MB/s, 17.3MB/s  ]
data/train-00000-of-00002.parquet: downloading bytes: 100% 205M/205M [00:03<00:00, 59.2MB/s, 17.6MB/s  ]
data/train-00000-of-00002.parquet: reconstructing file: 100% 206M/206M [00:03<00:00, 59.5MB/s, 18.6MB/s  ]
Generating train split: 100% 2717146/2717146 [00:04<00:00, 588842.09 examples/s]
[i] Columns: ['turkish', 'ottoman']  |  Rows: 2,717,146
[i] Raw data saved: out/step1/raw.parquet

========== SUMMARY ==========
total_rows                                 2717146
empty_ottoman                              0
empty_turkish                              0
dup_exact_pairs                            0
dup_exact_pairs_pct                        0.0
dup_ottoman_only                           8190
dup_tu

In [10]:
%cd /content/OttoLatin

!python scripts/step2_clean_split.py --no_overlap_check

/content/OttoLatin
[i] Input: 2,717,146 rows
[i] 2,393,924 Ottoman rows changed by normalization
[i] Filters:
    - empty                   0 dropped -> remaining 2,717,146
    - too_long              677 dropped -> remaining 2,716,469
    - few_letters        67,148 dropped -> remaining 2,649,321
    - digit_mismatch         65 dropped -> remaining 2,649,256
    - len_ratio               1 dropped -> remaining 2,649,255
    - duplicate              35 dropped -> remaining 2,649,220
[i] Sentence-level decontamination (removing test/dev sentences from train)...
    removed 934 rows from dev, 178,413 rows from train
[i] train        2,417,982 rows -> data/processed/train.jsonl
[i] dev             25,357 rows -> data/processed/dev.jsonl
[i] test            26,534 rows -> data/processed/test.jsonl
[i] train_small    300,000 rows -> data/processed/train_small.jsonl

[i] Report: out/step2/clean_report.json


In [14]:
%cd /content/OttoLatin

!zip -j train_small_backup.zip data/processed/train_small.jsonl


/content/OttoLatin
  adding: train_small.jsonl (deflated 63%)


In [15]:
from google.colab import files

files.download("/content/OttoLatin/train_small_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
%cd /content/OttoLatin

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/byt5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

dataset = load_dataset(
    "json",
    data_files={
        "train": "data/processed/train_small.jsonl",
        "validation": "data/processed/dev.jsonl",
    },
)

print("Train rows:", len(dataset["train"]))
print("Validation rows:", len(dataset["validation"]))
print("Model loaded:", MODEL_NAME)

example = dataset["train"][0]
print("Ottoman:", example["ottoman"])
print("Turkish:", example["turkish"])
print("Token count:", len(tokenizer(example["ottoman"])["input_ids"]))

/content/OttoLatin


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train rows: 300000
Validation rows: 25357
Model loaded: google/byt5-small
Ottoman: ١٦٩٠ طوغوملولر. اوخ ایی اولاجق!
Turkish: 1690 doğumlular. Oh iyi olacak!
Token count: 57


In [17]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

MAX_LENGTH = 512

def preprocess(batch):
    inputs = tokenizer(
        batch["ottoman"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["turkish"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

print(tokenized_dataset)
print("Tokenization completed.")

Tokenizing dataset:   0%|          | 0/300000 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/25357 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 300000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 25357
    })
})
Tokenization completed.


In [20]:
import os
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

OUTPUT_DIR = "/content/OttoLatin/checkpoints/byt5_short"

model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=1000,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    predict_with_generate=False,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting ByT5 training...")
trainer.train()
print("ByT5 training completed.")

Starting ByT5 training...


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
250,1.238835,0.161199
500,0.712388,0.109574
750,0.578162,0.091411
1000,0.528613,0.084411


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ByT5 training completed.


In [21]:
import json
import os
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = trainer.model.to(device)
model.config.use_cache = True
model.eval()

PRED_DIR = "/content/OttoLatin/preds/byt5"
os.makedirs(PRED_DIR, exist_ok=True)

SETS = ["test_2k", "riqa", "naskh", "ot2tr"]
BATCH_SIZE = 8

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

for set_name in SETS:
    input_path = f"/content/OttoLatin/data/eval/{set_name}.jsonl"
    output_path = f"{PRED_DIR}/{set_name}.jsonl"
    rows = load_jsonl(input_path)
    predictions = []

    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        texts = [row["ottoman"] for row in batch]

        encoded = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=512,
                num_beams=1,
                do_sample=False,
            )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
        )

        for row, prediction in zip(batch, decoded):
            predictions.append({
                "id": str(row["id"]),
                "prediction": prediction.strip().split("\n")[0].strip(),
            })

        if start % (BATCH_SIZE * 50) == 0:
            print(f"{set_name}: {min(start + BATCH_SIZE, len(rows))}/{len(rows)}")

    with open(output_path, "w", encoding="utf-8") as f:
        for prediction in predictions:
            f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

    print(f"Saved {len(predictions)} predictions: {output_path}")

print("All ByT5 predictions completed.")

test_2k: 8/2000
test_2k: 408/2000
test_2k: 808/2000
test_2k: 1208/2000
test_2k: 1608/2000
Saved 2000 predictions: /content/OttoLatin/preds/byt5/test_2k.jsonl
riqa: 8/1575
riqa: 408/1575
riqa: 808/1575
riqa: 1208/1575
Saved 1575 predictions: /content/OttoLatin/preds/byt5/riqa.jsonl
naskh: 8/376
Saved 376 predictions: /content/OttoLatin/preds/byt5/naskh.jsonl
ot2tr: 8/2358
ot2tr: 408/2358
ot2tr: 808/2358
ot2tr: 1208/2358
ot2tr: 1608/2358
ot2tr: 2008/2358
Saved 2358 predictions: /content/OttoLatin/preds/byt5/ot2tr.jsonl
All ByT5 predictions completed.


In [22]:
%cd /content/OttoLatin

import json
import os
import zipfile

sets = {
    "test_2k": 2000,
    "riqa": 1575,
    "naskh": 376,
    "ot2tr": 2358,
}

pred_dir = "preds/byt5"

for set_name, expected_count in sets.items():
    path = f"{pred_dir}/{set_name}.jsonl"

    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]

    assert len(rows) == expected_count
    assert all("id" in row and "prediction" in row for row in rows)
    assert len({row["id"] for row in rows}) == expected_count

    print(f"{set_name}: {len(rows)} rows - OK")

zip_path = "preds_byt5.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for set_name in sets:
        path = f"{pred_dir}/{set_name}.jsonl"
        archive.write(path, arcname=f"byt5/{set_name}.jsonl")

print(f"Created: {zip_path}")

/content/OttoLatin
test_2k: 2000 rows - OK
riqa: 1575 rows - OK
naskh: 376 rows - OK
ot2tr: 2358 rows - OK
Created: preds_byt5.zip


In [23]:
from google.colab import files

files.download("/content/OttoLatin/preds_byt5.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>